# Steel Defect Segmentation — YOLO26s-seg training (Colab)

Trains `yolo26s-seg` (imgsz=1024) on the Severstal Steel Defect Detection dataset converted to YOLO-seg format.

**Before Runtime → Run all:**
1. Runtime → Change runtime type → **GPU** (T4 works, A100 is faster).
2. Set `REPO_URL` in the config cell below to this project's GitHub URL.
3. Kaggle credentials — get one from kaggle.com → profile → Settings → API. Any ONE of these works:
   - **Colab Secret `KAGGLE_API_TOKEN`** (current Kaggle API token — recommended, key icon in left sidebar, enable notebook access); or
   - **Colab Secrets `KAGGLE_USERNAME` + `KAGGLE_KEY`** (legacy key pair); or
   - upload the token as `MyDrive/kaggle_access_token.txt` (just the raw token string); or
   - upload a legacy `kaggle.json` to Google Drive root (`MyDrive/kaggle.json`).
4. You must have joined the [Severstal competition](https://www.kaggle.com/competitions/severstal-steel-defect-detection) and accepted its rules.

Checkpoints are written to Drive every epoch, so if the session disconnects just **Run all again — training auto-resumes from `last.pt`**.

When training finishes, `best.pt` and `results.csv` are copied to `MyDrive/steel-defect-segmentation/final/`. Download `best.pt` into the repo's `weights/` folder for Phase 2 (evaluation / export / demo).

In [1]:
# ---- config ----
REPO_URL = "https://github.com/tun0000/steel-defect-segmentation.git"
RUN_NAME = "yolo26s-seg-1024"
DRIVE_ROOT = "/content/drive/MyDrive/steel-defect-segmentation"
IMGSZ = 1024
EPOCHS = 100
SEED = 42

assert REPO_URL != "TODO_SET_ME", "Set REPO_URL to the project's GitHub URL first"

In [2]:
# ---- GPU check ----
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime -> Change runtime type -> GPU"

Tue Jul  7 10:06:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# ---- mount Drive (checkpoints live here so disconnects are safe) ----
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ---- Kaggle credentials: current API token first, legacy formats as fallback ----
import os, shutil
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)


def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


api_token = _secret("KAGGLE_API_TOKEN")
username, key = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
drive_token = Path("/content/drive/MyDrive/kaggle_access_token.txt")
drive_json = Path("/content/drive/MyDrive/kaggle.json")

if api_token:
    os.environ["KAGGLE_API_TOKEN"] = api_token
    print("Kaggle credentials: KAGGLE_API_TOKEN Colab Secret")
elif username and key:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = username, key
    print("Kaggle credentials: KAGGLE_USERNAME/KAGGLE_KEY Colab Secrets (legacy)")
elif drive_token.exists():
    shutil.copy(drive_token, kaggle_dir / "access_token")
    (kaggle_dir / "access_token").chmod(0o600)
    print("Kaggle credentials: MyDrive/kaggle_access_token.txt")
elif drive_json.exists():
    shutil.copy(drive_json, kaggle_dir / "kaggle.json")
    (kaggle_dir / "kaggle.json").chmod(0o600)
    print("Kaggle credentials: MyDrive/kaggle.json (legacy)")
else:
    raise RuntimeError(
        "No Kaggle credentials found. Add a KAGGLE_API_TOKEN Colab Secret (recommended), "
        "or KAGGLE_USERNAME + KAGGLE_KEY Colab Secrets, or upload kaggle_access_token.txt "
        "/ kaggle.json to MyDrive root."
    )

Kaggle credentials: KAGGLE_API_TOKEN Colab Secret


In [5]:
# ---- clone repo + install deps ----
!rm -rf /content/repo && git clone {REPO_URL} /content/repo
%pip install -q "ultralytics>=8.4.0" "kaggle>=2.2"

Cloning into '/content/repo'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 52 (delta 16), reused 50 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 838.06 KiB | 13.30 MiB/s, done.
Resolving deltas: 100% (16/16), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.8/243.8 kB 25.9 MB/s eta 0:00:00


In [6]:
# ---- download competition data (not redistributable -> always pulled from Kaggle) ----
!kaggle competitions download -c severstal-steel-defect-detection -p /content/severstal/raw
!unzip -q -o /content/severstal/raw/severstal-steel-defect-detection.zip -d /content/severstal/raw
!ls /content/severstal/raw

100% 1.57G/1.57G [01:17<00:00, 21.6MB/s]

sample_submission.csv		      test_images  train_images
severstal-steel-defect-detection.zip  train.csv


In [7]:
# ---- convert RLE -> YOLO-seg with the exact same script used locally ----
!python /content/repo/scripts/convert_severstal_to_yolo.py \
    --raw-dir /content/severstal/raw \
    --out-dir /content/severstal/yolo \
    --reports-dir /content/reports \
    --seed {SEED}

RLE round-trip check passed for all 7095 annotations.
  processed 1000 images...
  processed 2000 images...
  processed 3000 images...
  processed 4000 images...
  processed 5000 images...
  processed 6000 images...
  processed 7000 images...
# Severstal -> YOLO-seg dataset stats

- seed: 42, val_frac: 0.1, neg_ratio: 0.1, min_area: 16px
- source images: 12568 total, 6666 with defects, 5902 defect-free
- kept: 6598 train / 734 val images (negatives: 599 train / 67 val)
- RLE round-trip: PASSED (7095 annotations)
- degenerate polygons skipped (<3 points): 3

| class | train imgs | val imgs | train instances | val instances | dropped <min_area | instances w/ holes |
|-------|-----------:|---------:|----------------:|--------------:|------------------:|-------------------:|
| defect_1 | 807 | 90 | 2789 | 293 | 0 | 17 |
| defect_2 | 222 | 25 | 291 | 30 | 0 | 1 |
| defect_3 | 4636 | 514 | 13132 | 1479 | 34 | 155 |
| defect_4 | 721 | 80 | 1687 | 210 | 10 | 55 |

Dataset written to /content/s

In [8]:
# ---- train (auto-resumes from Drive checkpoint if the session died mid-run) ----
from pathlib import Path
from ultralytics import YOLO

run_dir = Path(DRIVE_ROOT) / "runs" / RUN_NAME
last_ckpt = run_dir / "weights" / "last.pt"

if last_ckpt.exists():
    print(f"Resuming from {last_ckpt}")
    model = YOLO(str(last_ckpt))
    results = model.train(resume=True)
else:
    model = YOLO("yolo26s-seg.pt")
    results = model.train(
        data="/content/severstal/yolo/data.yaml",
        imgsz=IMGSZ,
        epochs=EPOCHS,
        patience=20,
        batch=64,          # tuned for A100 80GB (High-RAM). Use 32 on A100 40GB, lower on T4/L4.
        seed=SEED,
        project=str(run_dir.parent),
        name=RUN_NAME,
        exist_ok=True,
        # steel-strip-specific augmentation
        degrees=0.0,       # orientation is fixed on the production line
        fliplr=0.5,
        flipud=0.5,        # vertical flip is physically plausible for strips
        copy_paste=0.3,    # seg-only aug, helps rare classes
        hsv_h=0.0,         # images are effectively grayscale
        hsv_s=0.0,
        hsv_v=0.4,
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/severstal/yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=

In [9]:
# ---- final validation with best.pt ----
from ultralytics import YOLO

best_ckpt = run_dir / "weights" / "best.pt"
metrics = YOLO(str(best_ckpt)).val(data="/content/severstal/yolo/data.yaml", imgsz=IMGSZ)
print(f"mask mAP50    : {metrics.seg.map50:.4f}")
print(f"mask mAP50-95 : {metrics.seg.map:.4f}")

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLO26s-seg summary (fused): 139 layers, 10,366,888 parameters, 0 gradients, 34.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2455.1±797.8 MB/s, size: 115.2 KB)
val: Scanning /content/severstal/yolo/labels/val.cache... 734 images, 67 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 734/734 219.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 9.1it/s 5.1s
                   all        734       2012      0.692      0.591      0.659      0.343      0.652      0.553      0.587      0.233
              defect_1         90        293      0.656      0.512      0.599      0.265      0.599      0.464      0.537      0.173
              defect_2         25         30      0.688      0.633      0.707      0.332      0.619      0.567      0.544      0.181
              defect_

In [10]:
# ---- copy final artifacts to a stable Drive location ----
import shutil
from pathlib import Path

final_dir = Path(DRIVE_ROOT) / "final"
final_dir.mkdir(parents=True, exist_ok=True)
for artifact in ("weights/best.pt", "results.csv", "args.yaml"):
    src = run_dir / artifact
    if src.exists():
        shutil.copy2(src, final_dir / src.name)
        print(f"copied {src} -> {final_dir / src.name}")
print("Done. Download best.pt into the repo's weights/ folder for Phase 2.")

copied /content/drive/MyDrive/steel-defect-segmentation/runs/yolo26s-seg-1024/weights/best.pt -> /content/drive/MyDrive/steel-defect-segmentation/final/best.pt
copied /content/drive/MyDrive/steel-defect-segmentation/runs/yolo26s-seg-1024/results.csv -> /content/drive/MyDrive/steel-defect-segmentation/final/results.csv
copied /content/drive/MyDrive/steel-defect-segmentation/runs/yolo26s-seg-1024/args.yaml -> /content/drive/MyDrive/steel-defect-segmentation/final/args.yaml
Done. Download best.pt into the repo's weights/ folder for Phase 2.
